# AI MCQ Pipeline — Kaggle Edition (Multi-Agent Sandbox)

Kiến trúc Đồ thị đa tác tử (Multi-Agent Graph) kết hợp LLM và Python Sandbox để đạt độ chính xác >60%. Tối ưu hóa riêng cho môi trường Kaggle (T4 x2 hoặc P100).


In [ ]:
# Cài đặt Unsloth phiên bản tối ưu cho Kaggle (Sử dụng kaggle-new để tương thích PyTorch 2.4/2.5)
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install pandas tqdm


In [ ]:
# ===========================================================================
# 1. KHỞI TẠO ĐỘC QUYỀN UNSLOTH (BẮT BUỘC TRÊN CÙNG ĐỂ CHỐNG OOM ATTENTION)
# ===========================================================================
from unsloth import FastLanguageModel

import torch
if not hasattr(torch, "float8_e8m0fnu"):
    setattr(torch, "float8_e8m0fnu", torch.float32)

import transformers
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
if not hasattr(PreTrainedTokenizerBase, "all_special_tokens_extended"):
    PreTrainedTokenizerBase.all_special_tokens_extended = property(
        lambda self: self.all_special_tokens
    )

import nest_asyncio
nest_asyncio.apply()

import os
import re
import json
import time
import csv
import subprocess
import gc
import sys
from collections import Counter
from types import ModuleType
from typing import List, Dict, Any, TypedDict, Literal
import pandas as pd
from tqdm import tqdm
from pydantic import BaseModel, Field, ValidationError

def patch_pyairports():
    if "pyairports" not in sys.modules:
        dummy = ModuleType("pyairports")
        dummy_airports = ModuleType("pyairports.airports")
        dummy_airports.AIRPORT_LIST = []
        dummy.airports = dummy_airports
        sys.modules["pyairports"] = dummy
        sys.modules["pyairports.airports"] = dummy_airports

patch_pyairports()

# ===========================================================================
# 2. CẤU HÌNH TOÀN CỤC & PYDANTIC SCHEMAS
# ===========================================================================
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 4096  # Nới rộng dải ngữ cảnh lên 4096 để đọc văn bản Đọc hiểu dài

def get_dirs():
    if 'DATA_DIR' in globals() and 'OUTPUT_DIR' in globals():
        return globals()['DATA_DIR'], globals()['OUTPUT_DIR']
    if os.path.exists("/data"):
        return "/data", "/output"
    return ".", "./output"

LETTER_MAP = {i: chr(65 + i) for i in range(26)}

class RouterSchema(BaseModel):
    route: Literal["FAST_QA", "READING", "CODEABLE"] = Field(
        description="Phân luồng xử lý chính xác cho câu hỏi."
    )

class AnswerSchema(BaseModel):
    reasoning: str = Field(description="Suy luận ngắn gọn từng bước.")
    answer: str = Field(description="Chữ cái viết hoa duy nhất của đáp án đúng.")

# ===========================================================================
# 3. ĐỊNH NGHĨA TRẠNG THÁI TOÀN CỤC (GRAPH STATE)
# ===========================================================================
class GraphState(TypedDict):
    questions: List[dict]
    choice_counts: List[int]
    routes: List[str]
    execution_codes: List[str]
    sandbox_results: List[dict]
    final_answers: List[str]

# ===========================================================================
# 4. CÁC HÀM TIỆN ÍCH HỆ THỐNG
# ===========================================================================
def cleanup_vram():
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(1)

def format_choices(choices: list) -> str:
    return "\n".join(f"{LETTER_MAP[i]}. {c}" for i, c in enumerate(choices))

def parse_answer_to_letter(text: str) -> str:
    if not text:
        return "A"
    
    # Check if SUCCESS_MATCH
    if text.startswith("SUCCESS_MATCH:"):
        parts = text.split(":")
        if len(parts) >= 2:
            letter = parts[1].strip().upper()
            if len(letter) == 1 and letter in "ABCDEFGHIJKLMNOQPRSTUVWXYZ":
                return letter

    # Try parsing as JSON first
    text_clean = text.strip()
    json_match = re.search(r'```json\s*(.*?)\s*```', text_clean, re.DOTALL)
    if json_match:
        text_clean = json_match.group(1).strip()
    else:
        start_idx = text_clean.find('{')
        end_idx = text_clean.rfind('}')
        if start_idx != -1 and end_idx != -1:
            text_clean = text_clean[start_idx:end_idx+1]
            
    try:
        data = json.loads(text_clean)
        if isinstance(data, dict):
            ans = data.get("answer", "").strip().upper()
            if ans and len(ans) == 1 and ans in "ABCDEFGHIJKLMNOQPRSTUVWXYZ":
                return ans
            for k in ["choice", "result", "prediction"]:
                ans = data.get(k, "").strip().upper()
                if ans and len(ans) == 1 and ans in "ABCDEFGHIJKLMNOQPRSTUVWXYZ":
                    return ans
    except Exception:
        pass

    # Regex fallback
    match = re.search(r'\*\*([A-Z])\*\*', text)
    if match:
        return match.group(1)
        
    match = re.search(r'[đĐ]áp\s*án\s*(?:là|:)?\s*([A-Z])\b', text)
    if match:
        return match.group(1)
        
    match = re.search(r'[aA]nswer\s*(?:is|:)?\s*([A-Z])\b', text)
    if match:
        return match.group(1)
        
    matches = re.findall(r'\b([A-Z])\b', text)
    if matches:
        return matches[-1]

    matches = re.findall(r'([A-Z])', text)
    if matches:
        return matches[-1]

    return "A"

def save_pipeline_checkpoint(checkpoint_path: str, state: GraphState):
    with open(checkpoint_path, 'w', encoding='utf-8') as f:
        json.dump({
            "routes": state["routes"],
            "execution_codes": state["execution_codes"],
            "sandbox_results": state["sandbox_results"],
            "final_answers": state["final_answers"]
        }, f, ensure_ascii=False, indent=4)

def load_questions_pipeline():
    data_dir, _ = get_dirs()
    candidates = []
    for f in os.listdir(data_dir):
        if f.endswith('.json') and any(k in f.lower() for k in ['test', 'public', 'private']):
            candidates.append((os.path.join(data_dir, f), 'json'))
        elif f.endswith('.csv') and any(k in f.lower() for k in ['test', 'public', 'private']):
            candidates.append((os.path.join(data_dir, f), 'csv'))

    if not candidates:
        raise FileNotFoundError(f"Không tìm thấy file dữ liệu đầu vào hợp lệ tại {data_dir}!")

    target_file, file_type = candidates[0]
    if any(c[1] == 'csv' for c in candidates):
        target_file, file_type = [c for c in candidates if c[1] == 'csv'][0]

    print(f"[Hệ thống] Đang tiến hành nạp dữ liệu từ file: {target_file}")

    if file_type == 'json':
        with open(target_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    else:
        df = pd.read_csv(target_file)
        questions = []
        for _, row in df.iterrows():
            choices = []
            if 'choices' in df.columns:
                choices = json.loads(row['choices'])
            else:
                for col in [chr(65+i) for i in range(26)]:
                    if col in df.columns and pd.notna(row.get(col)):
                        choices.append(str(row[col]))
            questions.append({
                "qid": str(row["qid"]),
                "question": str(row["question"]),
                "choices": choices
            })
        return questions

def extract_numbers_from_text(text: str) -> list:
    text = text.replace(',', '')
    fraction_pattern = r'(-?\b\d+)/(\d+\b)'
    fractions = re.findall(fraction_pattern, text)
    nums = []
    for num, denom in fractions:
        try:
            nums.append(float(num) / float(denom))
        except ZeroDivisionError:
            pass
    text_clean = re.sub(fraction_pattern, '', text)
    decimal_pattern = r'-?\b\d+\.?\d*'
    raw_nums = re.findall(decimal_pattern, text_clean)
    for n in raw_nums:
        try:
            nums.append(float(n))
        except ValueError:
            pass
    return nums

def is_numeric_choices(choices: list) -> bool:
    for choice in choices:
        nums = extract_numbers_from_text(choice)
        if len(nums) != 1:
            return False
    return True

def clean_text_for_match(text: str) -> str:
    text = text.lower().strip()
    return re.sub(r'[\$\{\}\\\^_\(\)\*\/\\+\-\=\s\[\]\,\.]', '', text)

def find_closest_choice_info(python_output: str, choices: list) -> tuple:
    py_output_clean = clean_text_for_match(python_output)
    if py_output_clean and bool(re.search(r'[a-z]', py_output_clean)):
        for i, choice in enumerate(choices):
            choice_clean = clean_text_for_match(choice)
            if py_output_clean == choice_clean or py_output_clean in choice_clean or choice_clean in py_output_clean:
                if len(py_output_clean) >= 2 or py_output_clean == choice_clean:
                    return LETTER_MAP[i], 0.0, None

    if not is_numeric_choices(choices):
        return None, float('inf'), None

    code_nums = extract_numbers_from_text(python_output)
    if not code_nums:
        return None, float('inf'), None

    target_val = code_nums[-1]
    choices_nums = {}
    for i, choice in enumerate(choices):
        nums = extract_numbers_from_text(choice)
        if nums:
            choices_nums[LETTER_MAP[i]] = nums

    best_letter, min_relative_diff = None, float('inf')
    for letter, vals in choices_nums.items():
        for val in vals:
            denom = max(abs(val), 1.0)
            diff = abs(target_val - val) / denom
            if diff < min_relative_diff:
                min_relative_diff = diff
                best_letter = letter

    return best_letter, min_relative_diff, target_val

def execute_python_code(code: str, timeout_sec: int = 5) -> dict:
    temp_file = "temp_sandbox.py"
    with open(temp_file, "w", encoding="utf-8") as f:
        f.write(code)
    try:
        res = subprocess.run(
            [sys.executable, temp_file], capture_output=True, text=True, timeout=timeout_sec
        )
        return {"success": res.returncode == 0, "stdout": res.stdout, "stderr": res.stderr}
    except subprocess.TimeoutExpired:
        return {"success": False, "stdout": "", "stderr": f"Lỗi: Quá thời gian thực thi (>{timeout_sec} giây)."}
    finally:
        if os.path.exists(temp_file):
            try:
                os.remove(temp_file)
            except:
                pass

# ===========================================================================
# 5. LÕI INFERENCE PHÒNG THỦ KHỬ SẠCH RÁC ACTIVATION VRAM
# ===========================================================================
def unsloth_json_batch_inference(model, tokenizer, prompts: List[str], max_new_tokens: int = 256, temperature: float = 0.2, node_batch_size: int = 32) -> List[str]:
    results = []
    total_prompts = len(prompts)

    for i in range(0, total_prompts, node_batch_size):
        end_idx = min(i + node_batch_size, total_prompts)
        batch = prompts[i : end_idx]
        inputs = tokenizer(text=batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to("cuda")

        gen_kwargs = {
            "max_new_tokens": max_new_tokens,
            "use_cache": True
        }

        if temperature == 0.0:
            gen_kwargs["do_sample"] = False
        else:
            gen_kwargs["do_sample"] = True
            gen_kwargs["temperature"] = temperature

        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)

        for j, output in enumerate(outputs):
            input_len = inputs.input_ids[j].shape[0]
            generated_tokens = output[input_len:]
            decoded_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            results.append(decoded_text.strip())

        del inputs
        del outputs
        torch.cuda.empty_cache()

    return results

# ===========================================================================
# 6. THIẾT KẾ CHI TIẾT CÁC NODE AGENT TRÊN ĐỒ THỊ (MỖI BATCH LƯU CHECKPOINT)
# ===========================================================================

# --- NODE 1: LLM ROUTER NODE ---
def llm_router_node(state: GraphState, model, tokenizer, checkpoint_path: str) -> GraphState:
    print("\n" + "="*75 + "\n[Node 1]: Khởi chạy LLM Router Agent kết hợp trích xuất Metadata\n" + "="*75)

    choice_counts = [len(q["choices"]) for q in state["questions"]]
    state["choice_counts"] = choice_counts

    node_batch_size = 10
    total_questions = len(state["questions"])

    for i in range(0, total_questions, node_batch_size):
        end_idx = min(i + node_batch_size, total_questions)

        if all(state["routes"][k] != "" for k in range(i, end_idx)):
            continue

        print(f"      [Router Progress] Đang xử lý câu thứ {i + 1} đến {end_idx} / Tổng {total_questions} câu...")
        batch_questions = state["questions"][i:end_idx]
        prompts = []
        for q in batch_questions:
            prompt = tokenizer.apply_chat_template([
                {
                    "role": "system",
                    "content": (
                        "Bạn là bộ định tuyến dữ liệu siêu tốc của hệ thống LangGraph. Nhiệm vụ của bạn là phân loại câu hỏi đầu vào vào một nhóm duy nhất.\n"
                        "Bắt buộc phải trả về dữ liệu cấu trúc dưới dạng JSON khớp với schema sau:\n"
                        '{"route": "FAST_QA" | "READING" | "CODEABLE"}\n\n'
                        "Quy tắc phân loại:\n"
                        "1. 'READING': Nếu câu hỏi là đọc hiểu văn bản dài, có các đoạn thông tin hoặc đoạn trích lịch sử/pháp luật dài phức tạp.\n"
                        "2. 'CODEABLE': Nếu câu hỏi yêu cầu giải toán, tính công thức tài chính/kinh tế, tính số liệu lý/hóa cụ thể.\n"
                        "3. 'FAST_QA': Định nghĩa ngắn, tri thức nền hoặc kiểm tra sự thật đơn giản dưới 2000 ký tự.\n"
                        "Chỉ trả ra chuỗi JSON sạch. Tuyệt đối không viết thêm lời dẫn."
                    )
                },
                {"role": "user", "content": f"Câu hỏi: {q['question']}\nLựa chọn:\n{format_choices(q['choices'])}"}
            ], tokenize=False, add_generation_prompt=True)
            prompts.append(prompt)

        raw_outputs = unsloth_json_batch_inference(model, tokenizer, prompts, max_new_tokens=32, temperature=0.0, node_batch_size=node_batch_size)

        for j, raw_out in enumerate(raw_outputs):
            try:
                json_str = re.search(r'\{.*\}', raw_out, re.DOTALL).group(0)
                validated_data = RouterSchema.model_validate_json(json_str)
                state["routes"][i + j] = validated_data.route
            except Exception:
                state["routes"][i + j] = "FAST_QA"

        save_pipeline_checkpoint(checkpoint_path, state)

    counts = Counter(state["routes"])
    print(f"[Router Kết Quả] Phân luồng hoàn tất: {counts['FAST_QA']} câu Fast-QA, {counts['READING']} câu Đọc hiểu, {counts['CODEABLE']} câu Giải toán.")
    return state

# --- NODE 2: FAST-QA NODE ---
def fast_qa_agent_node(state: GraphState, model, tokenizer, checkpoint_path: str) -> GraphState:
    indices = [i for i, r in enumerate(state["routes"]) if r == "FAST_QA"]
    if not indices:
        return state

    print(f"\n[Node 2]: Khởi chạy Fast-QA Agent giải quyết {len(indices)} câu tri thức ngắn (Đường cao tốc)...")
    node_batch_size = 10
    total_prompts = len(indices)

    for i in range(0, total_prompts, node_batch_size):
        end_idx = min(i + node_batch_size, total_prompts)
        batch_indices = indices[i:end_idx]

        if all(state["final_answers"][idx] != "" for idx in batch_indices):
            continue

        print(f"      [LLM Batch Progress] Đang xử lý câu thứ {i + 1} đến {end_idx} / Tổng {total_prompts} câu...")
        prompts = []
        for idx in batch_indices:
            q = state["questions"][idx]

            # Few-shot mẫu ép khuôn tư duy ngắn gọn
            fast_qa_few_shots = (
                "Ví dụ mẫu:\n"
                "Câu hỏi: Thủ đô của Việt Nam là gì?\n"
                "Lựa chọn:\nA. TP. Hồ Chí Minh\nB. Hà Nội\n"
                "Đầu ra mẫu:\n"
                "```json\n"
                "{\n"
                "  \"reasoning\": \"Hà Nội là thủ đô hành chính chính thức của Việt Nam.\",\n"
                "  \"answer\": \"B\"\n"
                "}\n"
                "```\n\n"
            )

            prompt = tokenizer.apply_chat_template([
                {
                    "role": "system",
                    "content": (
                        "Bạn là chuyên gia trắc nghiệm tri thức. Hãy phân tích và chọn đáp án đúng duy nhất.\n"
                        "Bắt buộc phải trả về định dạng cấu trúc JSON sạch khớp chính xác với mẫu sau:\n"
                        '{"reasoning": "Suy luận ngắn gọn (độ dài tối đa 1 câu)", "answer": "Chữ cái viết hoa duy nhất (Ví dụ: A hoặc B hoặc C...)"}\n\n'
                        "QUY TẮC PHÒNG THỦ TRÀN TOKEN:\n"
                        "Phần 'reasoning' chỉ được viết đúng 1 câu duy nhất, đi thẳng vào bản chất tri thức nền. Tuyệt đối không giải thích dông dài, không viết ngoài khối JSON."
                    )
                },
                {
                    "role": "user",
                    "content": f"/no_think\n{fast_qa_few_shots}BÂY GIỜ HÃY GIẢI CÂU HỎI SAU VÀ TUÂN THỦ JSON THUẬN CÔ ĐỌNG:\nCâu hỏi: {q['question']}\nLựa chọn:\n{format_choices(q['choices'])}"
                }
            ], tokenize=False, add_generation_prompt=True)
            prompts.append(prompt)

        raw_outputs = unsloth_json_batch_inference(model, tokenizer, prompts, max_new_tokens=128, temperature=0.0, node_batch_size=node_batch_size)
        for j, raw_out in enumerate(raw_outputs):
            state["final_answers"][batch_indices[j]] = raw_out

        save_pipeline_checkpoint(checkpoint_path, state)
    return state

# --- NODE 3: READING COMPREHENSION NODE ---
def reading_comprehension_agent_node(state: GraphState, model, tokenizer, checkpoint_path: str) -> GraphState:
    indices = [i for i, r in enumerate(state["routes"]) if r == "READING"]
    if not indices:
        return state

    print(f"\n[Node 3]: Khởi chạy Reading Comprehension Agent bóc tách {len(indices)} câu văn bản ngữ cảnh dài...")
    node_batch_size = 10
    total_prompts = len(indices)

    for i in range(0, total_prompts, node_batch_size):
        end_idx = min(i + node_batch_size, total_prompts)
        batch_indices = indices[i:end_idx]

        if all(state["final_answers"][idx] != "" for idx in batch_indices):
            continue

        print(f"      [LLM Batch Progress] Đang xử lý câu thứ {i + 1} đến {end_idx} / Tổng {total_prompts} câu...")
        prompts = []
        for idx in batch_indices:
            q = state["questions"][idx]

            # Few-shot hướng dẫn trích xuất bối cảnh văn bản dài nhưng lập luận siêu ngắn
            reading_few_shots = (
                "Ví dụ mẫu:\n"
                "Câu hỏi: Giai đoạn Mạt Pháp bắt đầu khi nào theo tài liệu?\n"
                "Lựa chọn:\nA. 1000 năm\nB. 1500 năm\n"
                "Đầu ra mẫu:\n"
                "```json\n"
                "{\n"
                "  \"reasoning\": \"Tài liệu tại Đoạn 1 ghi nhận thời điểm Mạt Pháp bắt đầu là 1500 năm sau khi Thích Ca nhập niết bàn, khớp với phương án B.\",\n"
                "  \"answer\": \"B\"\n"
                "}\n"
                "```\n\n"
            )

            prompt = tokenizer.apply_chat_template([
                {
                    "role": "system",
                    "content": (
                        "Bạn là chuyên gia rà soát bẫy văn bản dài. Hãy đối chiếu chi tiết các tình huống, rà soát kỹ các từ phủ định (không phải, ngoại trừ, sai) để trích xuất thông tin.\n"
                        "Bắt buộc phải trả về định dạng cấu trúc JSON sạch khớp chính xác với mẫu sau:\n"
                        '{"reasoning": "Đối chiếu bối cảnh tài liệu (tối đa 1 đến 2 câu ngắn)", "answer": "Chữ cái đáp án viết hoa đúng nhất"}\n\n'
                        "QUY TẮC CHỐNG CẮT CỤT CHUỖI DO HẾT TOKEN:\n"
                        "Phần 'reasoning' phải viết cực kỳ cô đọng, đi thẳng vào việc chỉ rõ Đoạn/Tài liệu nào chứa từ khóa để chốt đáp án. Tuyệt đối không chép lại cả đoạn văn bản dài vào JSON."
                    )
                },
                {
                    "role": "user",
                    "content": f"/think\n{reading_few_shots}BÂY GIỜ HÃY ĐỐI CHIẾU VĂN BẢN VÀ GIẢI CÂU HỎI SAU:\nCâu hỏi: {q['question']}\nLựa chọn:\n{format_choices(q['choices'])}"
                }
            ], tokenize=False, add_generation_prompt=True)
            prompts.append(prompt)

        raw_outputs = unsloth_json_batch_inference(model, tokenizer, prompts, max_new_tokens=384, temperature=0.1, node_batch_size=node_batch_size)
        for j, raw_out in enumerate(raw_outputs):
            state["final_answers"][batch_indices[j]] = raw_out

        save_pipeline_checkpoint(checkpoint_path, state)
    return state

# --- NODE 4 & NODE 5: CODER AGENT & SANDBOX LOOP ---
def coder_and_sandbox_loop_node(state: GraphState, model, tokenizer, checkpoint_path: str) -> GraphState:
    indices = [i for i, r in enumerate(state["routes"]) if r == "CODEABLE"]
    if not indices:
        return state

    print(f"\n[Node 4 & 5]: Khởi chạy Coder Agent (Tham chiếu định dạng) & Sandbox Matching Host-side cho {len(indices)} câu...")

    node_batch_size = 8
    total_prompts = len(indices)

    # ===========================================================================
    # CHU TRÌNH 1: SINH MÃ THAM CHIẾU FORMAT ĐÁP ÁN (CẤM IN CHỮ CÁI ĐÁP ÁN NHÃN)
    # ===========================================================================
    for i in range(0, total_prompts, node_batch_size):
        end_idx = min(i + node_batch_size, total_prompts)
        batch_indices = indices[i:end_idx]

        if all(state["execution_codes"][idx] != "" for idx in batch_indices):
            continue

        print(f"      [LLM Code Gen Progress] Đang sinh mã căn chỉnh định dạng: Câu {i + 1} đến {end_idx} / Tối đa {total_prompts}...")
        current_prompts = []
        for idx in batch_indices:
            q = state["questions"][idx]

            # Hệ thống Few-shot ép mô hình luôn print ra giá trị PURE (Số thuần tuý hoặc công thức SymPy sạch)
            few_shots = (
                "Ví dụ 1 (Đề bài chứa số hoặc đơn vị kèm theo như %, g, kJ/mol, đô la):\n"
                "Câu hỏi: Tính khối lượng NaOH cần dùng để trung hòa 200ml dung dịch HCl 1M.\n"
                "Lựa chọn đáp án:\nA. 40g.\nB. 80g.\nC. 160g.\n"
                "Mã nguồn mẫu:\n"
                "```python\n"
                "try:\n"
                "    n_hcl = 0.2 * 1.0\n"
                "    m_naoh = n_hcl * 40.0\n"
                "    print(float(m_naoh))\n"
                "except:\n"
                "    print('ERROR')\n"
                "```\n\n"
                "Ví dụ 2 (Đề bài chứa công thức kí tự, phương trình đại số):\n"
                "Câu hỏi: Tìm nghiệm của phương trình vi phân đạo hàm bậc nhất B'(t) = -k*B(t) với điều kiện biên B(0)=B0.\n"
                "Lựa chọn đáp án:\nA. B0 * exp(-k * t)\nB. B0 / (1 + k * t)\n"
                "Mã nguồn mẫu:\n"
                "```python\n"
                "try:\n"
                "    import sympy\n"
                "    t, B0, k = sympy.symbols('t B0 k')\n"
                "    ket_qua = B0 * sympy.exp(-k * t)\n"
                "    print(str(ket_qua))\n"
                "except:\n"
                "    print('ERROR')\n"
                "```\n\n"
            )

            prompt = tokenizer.apply_chat_template([
                {
                    "role": "system",
                    "content": (
                        "Bạn là một chuyên gia lập trình Python tối giản, phụ trách xử lý các bài toán định lượng và logic đa ngành.\n"
                        "Nhiệm vụ: Hãy viết mã nguồn Python hoàn chỉnh để giải bài toán. Bạn được cung cấp danh sách phương án trắc nghiệm thực tế CHỈ ĐỂ tham khảo dạng kết quả số học hoặc kí tự đại số.\n\n"
                        "QUY TẮC ÉP KHUÔN ĐỊNH DẠNG TUYỆT ĐỐI KHÔNG GÂY LỖI CÚ PHÁP:\n"
                        "1. CẤM TUYỆT ĐỐI SỬ DỤNG KÝ TỰ DẤU THĂNG (#). Không viết comment, không tạo chuỗi hậu tố lặp lại dông dài.\n"
                        "2. CẤM TUYỆT ĐỐI IN RA CÁC KÝ TỰ NHÃN ĐÁP ÁN NHƯ 'A', 'B', 'C', 'D'...\n"
                        "3. QUY TẮC LỆNH PRINT() CUỐI CÙNG:\n"
                        "   - Nếu các phương án lựa chọn chứa số (kể cả số có kèm đơn vị như %, g, kJ/mol, đô la), lệnh print BẮT BUỘC chỉ được in ra duy nhất GIÁ TRỊ SỐ NGUYÊN HOẶC SỐ THỰC THUẦN TUÝ (Ví dụ: print(float(gia_tri))). Tuyệt đối không thêm chuỗi đơn vị vào lệnh print.\n"
                        "   - Nếu các phương án lựa chọn là biểu thức kí tự hoặc phương trình đại số (có chứa mã LaTeX $), lệnh print BẮT BUỘC phải in ra chuỗi biểu thức sạch sau khi gọi str(ket_qua) từ SymPy. Tuyệt đối không bọc kí tự $ vào lệnh print.\n"
                        "4. Toàn bộ mã nguồn phải bọc gọn trong cấu trúc phòng thủ try-except toàn cục. Nhánh except chỉ viết duy nhất: print('ERROR').\n"
                        "5. Chỉ xuất duy nhất khối mã nằm trong thẻ ```python và ```. Không viết lời dẫn."
                    )
                },
                {
                    "role": "user",
                    "content": f"/no_think\n{few_shots}BÂY GIỜ HÃY GIẢI CÂU HỎI SAU, CHÚ Ý PRINT KẾT QUẢ SỐ THUẦN TUÝ HOẶC CHUỖI BIỂU THỨC SẠCH KHÔNG CHỨA LỖI BIẾN SỐ:\nCâu hỏi: {q['question']}\nLựa chọn đáp án thực tế:\n{format_choices(q['choices'])}"
                }
            ], tokenize=False, add_generation_prompt=True)
            current_prompts.append(prompt)

        raw_codes = unsloth_json_batch_inference(model, tokenizer, current_prompts, max_new_tokens=512, temperature=0.0, node_batch_size=node_batch_size)
        for j, raw_code in enumerate(raw_codes):
            code_match = re.search(r'```python\s*(.*?)\s*```', raw_code, re.DOTALL)
            state["execution_codes"][batch_indices[j]] = code_match.group(1) if code_match else raw_code

        save_pipeline_checkpoint(checkpoint_path, state)

    # ===========================================================================
    # CHU TRÌNH 2: THỰC THI SANDBOX & SO KHỚP NGOẠI BIÊN (ĐÚNG 2 LẦN CHẠY CODE THỬ SAI)
    # ===========================================================================
    max_retries = 1  # retry = 0 (Lượt 1), retry = 1 (Lượt 2). Tổng cộng đúng 2 lần thử code.

    for attempt in range(max_retries + 1):
        active_indices = []
        for idx in indices:
            ans = state["final_answers"][idx]
            if ans.startswith("SUCCESS_MATCH") or (len(ans) == 1 and ans in "ABCDEFGHIJKLMNOQPRSTUVWXYZ") or "reasoning" in ans:
                continue
            active_indices.append(idx)

        if not active_indices:
            break

        print(f"   [Host-Side Matching] Tiến hành thực thi kiểm tra ngoại vi lượt thứ {attempt + 1} / Tối đa {max_retries + 1}...")

        next_active_indices = []
        correction_prompts_map = {}

        for curr_progress_idx, idx in enumerate(active_indices):
            code = state["execution_codes"][idx]
            print(f"      [Sandbox Running] ({curr_progress_idx + 1}/{len(active_indices)}) Đang chạy ngầm kiểm tra Host cho Index {idx}...")

            run_res = execute_python_code(code)
            state["sandbox_results"][idx] = run_res

            closest_letter, min_relative_diff, target_val = None, float('inf'), None
            if run_res['success']:
                closest_letter, min_relative_diff, target_val = find_closest_choice_info(run_res['stdout'], state["questions"][idx]['choices'])

            # Thành công nếu không lỗi runtime hệ thống và sai số dưới ngưỡng làm tròn 1%
            if run_res['success'] and closest_letter is not None and min_relative_diff < 0.01:
                state["final_answers"][idx] = f"SUCCESS_MATCH:{closest_letter}:{run_res['stdout'].strip()}"
            else:
                if not run_res['success']:
                    err_msg = f"Mã nguồn bị lỗi Runtime hệ thống.\nChi tiết log: {run_res['stderr'].strip()}"
                elif target_val is None and run_res['stdout'].strip() == "ERROR":
                    err_msg = f"Mã nguồn bị crash nhảy vào khối ngoại lệ 'except' do lỗi cú pháp hoặc gọi sai tên biến trong hàm print."
                elif target_val is None:
                    err_msg = f"Mã nguồn chạy thành công nhưng không in ra kết quả định dạng khớp với dải đáp án trắc nghiệm (stdout thu được: '{run_res['stdout'].strip()}')."
                else:
                    err_msg = f"Mã nguồn chạy thành công xuất ra số '{target_val}'. Tuy nhiên giá trị này lệch quá xa dải số của đề bài (Sai số nhỏ nhất đo được là {min_relative_diff*100:.2f}% tại phương án {closest_letter})."

                # TRƯỜNG HỢP CÒN LƯỢT THỬ (Mới chạy xong lần 1): Tạo prompt để LLM sửa đổi code
                if attempt < max_retries:
                    cor_prompt = tokenizer.apply_chat_template([
                        {
                            "role": "system",
                            "content": (
                                "Bạn là một chuyên gia sửa lỗi và tối ưu hóa mã nguồn Python đa ngành.\n"
                                "Nhiệm vụ: Viết lại một script Python hoàn chỉnh mới, sửa đổi logic tính toán hoặc sửa triệt để lỗi gọi tên biến/cú pháp trong hàm print để xuất ra kết quả thô chính xác.\n\n"
                                "QUY TẮC BẮT BUỘC:\n"
                                "1. CẤM TUYỆT ĐỐI SỬ DỤNG KÝ TỰ DẤU THĂNG (#). Không bình luận, không giải thích.\n"
                                "2. CẤM in ra các chữ cái nhãn đáp án A, B, C, D... Chỉ tinh chỉnh lệnh print xuất ra định dạng thô.\n"
                                "3. Kiểm tra kĩ lưỡng tên biến bên trong lệnh print() xem có trùng khớp với biến đã khai báo phía trên không.\n"
                                "4. Chỉ xuất duy nhất khối mã nằm trong thẻ ```python và ```."
                            )
                        },
                        {
                            "role": "user",
                            "content": (
                                f"/no_think\n"
                                f"Hãy sửa đổi và viết lại đoạn mã nguồn mới hoàn chỉnh để tính toán chính xác và khớp định dạng chuỗi đề bài.\n\n"
                                f"Câu hỏi: {state['questions'][idx]['question']}\n"
                                f"Lựa chọn đáp án thực tế:\n{format_choices(state['questions'][idx]['choices'])}\n\n"
                                f"Thông báo lỗi thu được từ Host bên ngoài:\n{err_msg}\n\n"
                                f"Đoạn mã nguồn lỗi cần thay thế:\n{code}"
                            )
                        }
                    ], tokenize=False, add_generation_prompt=True)

                    correction_prompts_map[idx] = cor_prompt
                    next_active_indices.append(idx)

                # TRƯỜNG HỢP ĐÃ SAI ĐỦ 2 LẦN: Ngắt ngay lập tức, chuyển hướng sang tác vụ lập luận văn bản (Fallback)
                else:
                    print(f"      [🚨 Ngắt vòng lặp - Kích hoạt Node Suy luận Văn bản] Index {idx} đã thử sai 2 lần. Ép LLM giải đề ngược vết log lỗi...")
                    fallback_prompt = tokenizer.apply_chat_template([
                        {
                            "role": "system",
                            "content": (
                                "Bạn là chuyên gia giải đề trắc nghiệm có tư duy phản biện cao.\n"
                                "Thuật toán viết mã giải toán trước đó của hệ thống đã bị tính toán lệch số hoặc lỗi runtime sau nhiều lần thử.\n"
                                "Nhiệm vụ: Dựa vào câu hỏi, các phương án lựa chọn và chi tiết log lỗi của chương trình, hãy phân tích lập luận văn bản để tìm ra đáp án đúng cuối cùng.\n"
                                "Bắt buộc phải trả về định dạng cấu trúc JSON sạch khớp chính xác với schema sau:\n"
                                '{"reasoning": "Phân tích logic và suy luận biện giải đáp án đúng dựa vào vết sai số của code", "answer": "Chữ cái viết hoa đại diện đáp án chuẩn xác nhất"}\n'
                                "Tuyệt đối không giải thích thêm ngoài khối JSON này."
                            )
                        },
                        {
                            "role": "user",
                            "content": (
                                f"Câu hỏi: {state['questions'][idx]['question']}\n"
                                f"Lựa chọn:\n{format_choices(state['questions'][idx]['choices'])}\n\n"
                                f"Mã nguồn tính toán bị lỗi trước đó:\n{code}\n\n"
                                f"Log lỗi hệ thống ngoài Host thu được:\n{err_msg}"
                            )
                        }
                    ], tokenize=False, add_generation_prompt=True)

                    fallback_out = unsloth_json_batch_inference(model, tokenizer, [fallback_prompt], max_new_tokens=384, temperature=0.2, node_batch_size=10)
                    state["final_answers"][idx] = fallback_out[0]

        save_pipeline_checkpoint(checkpoint_path, state)

        if next_active_indices and attempt < max_retries:
            print(f"   [Vá lỗi Batch] Đang gọi LLM tái cấu trúc lại mã nguồn cho {len(next_active_indices)} câu hỏi...")
            cor_batch_size = 8
            cor_indices_list = list(next_active_indices)
            for c_i in range(0, len(cor_indices_list), cor_batch_size):
                c_end = min(c_i + cor_batch_size, len(cor_indices_list))
                sub_batch_indices = cor_indices_list[c_i:c_end]

                print(f"      [LLM Correction Progress] Đang vá lỗi câu thứ {c_i + 1} đến {c_end} / Tổng {len(cor_indices_list)} câu...", flush=True)
                prompts_batch = [correction_prompts_map[idx] for idx in sub_batch_indices]
                raw_cor_codes = unsloth_json_batch_inference(model, tokenizer, prompts_batch, max_new_tokens=512, temperature=0.1, node_batch_size=cor_batch_size)

                for j, raw_cor_code in enumerate(raw_cor_codes):
                    code_match = re.search(r'```python\s*(.*?)\s*```', raw_cor_code, re.DOTALL)
                    state["execution_codes"][sub_batch_indices[j]] = code_match.group(1) if code_match else raw_cor_code

                save_pipeline_checkpoint(checkpoint_path, state)
        else:
            break

    # ===========================================================================
    # CHU TRÌNH 3: CHẠY HỢP NHẤT PHIẾU BẦU (MAJORITY VOTING)
    # ===========================================================================
    print("   [Toán học Voting] Tiến hành hợp nhất kết quả logic số học thông qua LLM Majority Voting...")
    vote_batch_size = 2
    for v_i in range(0, len(indices), vote_batch_size):
        v_end = min(v_i + vote_batch_size, len(indices))
        batch_indices = indices[v_i:v_end]

        if all(len(state["final_answers"][idx]) == 1 and state["final_answers"][idx] in "ABCDEFGHIJKLMNOQPRSTUVWXYZ" for idx in batch_indices):
            continue

        voting_prompts = []
        for idx in batch_indices:
            q = state["questions"][idx]
            current_status = state["final_answers"][idx]

            if current_status.startswith("SUCCESS_MATCH"):
                _, closest_letter, stdout_val = current_status.split(":", 2)
                python_hint = f"\n[Kết quả chạy chương trình Python gợi ý]: {stdout_val}\n[Khớp phương án số học tương ứng]: {closest_letter}"
            elif "reasoning" in current_status:
                python_hint = f"\n[Lưu ý hệ thống]: Sandbox lỗi toán học. Kết quả phân tích suy luận văn bản dự phòng thu được: {current_status}"
            else:
                python_hint = "\n[Lưu ý hệ thống]: Không thể thực thi mã Python hoặc trích xuất suy luận văn bản chuẩn xác."

            v_prompt = tokenizer.apply_chat_template([
                {
                    "role": "system",
                    "content": (
                        "Bạn là chuyên gia toán học xuất sắc. Dựa trên câu hỏi, phương án trắc nghiệm và kết quả gợi ý định lượng được cung cấp, hãy chọn đáp án đúng nhất.\n"
                        "Bắt buộc phải trả về định dạng cấu trúc JSON sạch khớp chính xác với schema sau:\n"
                        '{"reasoning": "Suy luận logic toán học ngắn gọn", "answer": "Chữ cái lựa chọn đúng duy nhất"}\n'
                        "Tuyệt đối không giải thích thêm ngoài khối JSON."
                    )
                },
                {
                    "role": "user",
                    "content": f"/think\nCâu hỏi: {q['question']}\nLựa chọn:\n{format_choices(q['choices'])}\n{python_hint}"
                }
            ], tokenize=False, add_generation_prompt=True)
            voting_prompts.append(v_prompt)

        n_votes = 3
        all_votes_matrix = []
        for vote_round in range(n_votes):
            print(f"      [Voting Round] Đang lấy phiếu bầu lượt số {vote_round + 1} / {n_votes} cho câu thứ {v_i + 1} đến {v_end}...")
            temp_outs = unsloth_json_batch_inference(model, tokenizer, voting_prompts, max_new_tokens=256, temperature=0.3, node_batch_size=vote_batch_size)
            all_votes_matrix.append(temp_outs)

        for list_pos, idx in enumerate(batch_indices):
            votes_counter = Counter()
            for vote_round in range(n_votes):
                raw_text_ans = all_votes_matrix[vote_round][list_pos]
                parsed_letter = parse_answer_to_letter(raw_text_ans)
                votes_counter[parsed_letter] += 1.0

            if state["final_answers"][idx].startswith("SUCCESS_MATCH"):
                _, closest_letter, _ = state["final_answers"][idx].split(":", 2)
                if closest_letter:
                    votes_counter[closest_letter] += 2.0

            state["final_answers"][idx] = votes_counter.most_common(1)[0][0]

        save_pipeline_checkpoint(checkpoint_path, state)

    return state

# --- NODE 6: ANSWER VOTER & DYNAMIC MAPPER ---
def answer_voter_and_mapper_node(state: GraphState) -> GraphState:
    print("\n[Node 6]: Khởi chạy Answer Voter & Dynamic Mapper kiểm soát và ép định dạng dải đáp án trắc nghiệm...")
    mapped_answers = []
    total_answers = len(state["final_answers"])

    for idx, raw_ans in enumerate(state["final_answers"]):
        letter = parse_answer_to_letter(raw_ans) if len(raw_ans) > 1 else raw_ans

        max_allowed_index = state["choice_counts"][idx] - 1
        detected_index = ord(letter) - 65

        if (idx + 1) % 50 == 0 or (idx + 1) == total_answers:
            print(f"      [Mapper Progress] Đang chuẩn hóa dải phương án đến câu thứ {idx + 1} / {total_answers}...")

        if detected_index < 0 or detected_index > max_allowed_index:
            try:
                choices = state["questions"][idx]["choices"]
                best_match_letter = "A"
                max_sim_score = -1

                clean_target = clean_text_for_match(raw_ans)
                for i in range(len(choices)):
                    clean_comp = clean_text_for_match(choices[i])
                    sim_score = len(set(clean_target) & set(clean_comp))
                    if sim_score > max_sim_score:
                        max_sim_score = sim_score
                        best_match_letter = LETTER_MAP[i]
                mapped_answers.append(best_match_letter)
            except Exception:
                mapped_answers.append("A")
        else:
            mapped_answers.append(letter)

    state["final_answers"] = mapped_answers
    return state

# ===========================================================================
# 7. KHỞI CHẠY PIPELINE TUYẾN TÍNH AN TOÀN TUYỆT ĐỐI (MASTER CHECKPOINT)
# ===========================================================================
def main():
    start_time = time.time()
    cleanup_vram()

    _, output_dir = get_dirs()

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    try:
        raw_questions = load_questions_pipeline()
    except Exception as e:
        print(f"[Lỗi Hệ Thống] Không thể nạp tệp dữ liệu: {e}")
        return

    print(f"[Hệ thống] Đã nạp và nhận diện thành công toàn bộ {len(raw_questions)} câu hỏi kiểm tra.")

    state = GraphState(
        questions=raw_questions,
        choice_counts=[],
        routes=[""] * len(raw_questions),
        execution_codes=[""] * len(raw_questions),
        sandbox_results=[{"success": False, "stdout": "", "stderr": ""}] * len(raw_questions),
        final_answers=[""] * len(raw_questions)
    )

    choice_counts = [len(q["choices"]) for q in state["questions"]]
    state["choice_counts"] = choice_counts

    checkpoint_path = os.path.join(output_dir, "pipeline_checkpoint.json")

    # Nạp khôi phục trạng thái bộ nhớ gia tăng nếu có file
    if os.path.exists(checkpoint_path):
        print(f"[Checkpoint] Phát hiện tệp khôi phục tiến trình cũ tại {checkpoint_path}.")
        try:
            with open(checkpoint_path, 'r', encoding='utf-8') as f:
                saved = json.load(f)
                if "routes" in saved and len(saved["routes"]) == len(raw_questions):
                    state["routes"] = saved["routes"]
                if "execution_codes" in saved and len(saved["execution_codes"]) == len(raw_questions):
                    state["execution_codes"] = saved["execution_codes"]
                if "sandbox_results" in saved and len(saved["sandbox_results"]) == len(raw_questions):
                    state["sandbox_results"] = saved["sandbox_results"]
                if "final_answers" in saved and len(saved["final_answers"]) == len(raw_questions):
                    state["final_answers"] = saved["final_answers"]
            print("[Checkpoint] Nạp trạng thái thành công, tiếp tục thực thi từ điểm dừng gần nhất.")
        except Exception as e:
            print(f"[Checkpoint Lỗi] Không thể nạp checkpoint: {e}, khởi tạo tiến trình mới.")

    print("[Unsloth Engine] Đang khởi tạo mô hình nền tảng Qwen 2.5 7B (4-bit mode)...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype = None,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Vận hành luồng dữ liệu phân tầng qua các Node Agent kế thừa Checkpoint gia tăng
    state = llm_router_node(state, model, tokenizer, checkpoint_path)
    state = fast_qa_agent_node(state, model, tokenizer, checkpoint_path)
    state = reading_comprehension_agent_node(state, model, tokenizer, checkpoint_path)
    state = coder_and_sandbox_loop_node(state, model, tokenizer, checkpoint_path)

    del model
    del tokenizer
    cleanup_vram()

    state = answer_voter_and_mapper_node(state)

    output_csv = os.path.join(output_dir, "pred.csv")
    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["qid", "answer"])
        for q, ans in zip(state["questions"], state["final_answers"]):
            writer.writerow([q['qid'], ans if ans in "ABCDEFGHIJKLMNOQPRSTUVWXYZ" else "A"])

    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    elapsed = time.time() - start_time
    print(f"\n[HOÀN THÀNH TỐI CAO] Toàn bộ Đồ thị Multi-Agent Pipeline đã kết thúc chu kỳ xử lý an toàn trong {elapsed:.2f} giây!")
    print(f"[Xuất bài] Kết quả lưu tại: {output_csv}")

if __name__ == "__main__":
    main()


### Cấu hình Dữ Liệu & Chạy Pipeline (Kaggle)

**LƯU Ý:** Bạn cần nhấn nút **Add Input** ở góc phải màn hình Kaggle để đính kèm file dataset (json/csv) vào Notebook trước khi chạy.


In [ ]:
import os
import shutil

# Trên Kaggle, thư mục working là nơi có quyền ghi file
globals()['OUTPUT_DIR'] = "/kaggle/working/output"

# Tìm file json/csv trong thư mục /kaggle/input (nơi Kaggle mount dataset)
input_dir = "/kaggle/input"
found_file = None

for root, dirs, files in os.walk(input_dir):
    for file in files:
        if file.endswith('.json') or file.endswith('.csv'):
            found_file = os.path.join(root, file)
            break
    if found_file:
        break

if found_file:
    print(f"Đã tìm thấy file dữ liệu tại: {found_file}")
    # Đặt DATA_DIR trỏ tới thư mục chứa file đó
    globals()['DATA_DIR'] = os.path.dirname(found_file)
else:
    print("⚠️ CẢNH BÁO: Chưa tìm thấy file dữ liệu trong /kaggle/input. Bạn đã Add Data chưa?")
    globals()['DATA_DIR'] = "/kaggle/working"

# Kích hoạt hệ thống Multi-Agent
main()

print("\nHoàn tất! Bạn có thể xem kết quả trong thư mục /kaggle/working/output ở cột bên phải.")

